# Orders dataset cleaning 

In [ ]:
import pandas as pd
import numpy as np

In [ ]:


orders = pd.read_csv(r"C:\Users\MAINAK\Desktop\Data-Science\projects\MarketLens\data\raw\olist_orders_dataset.csv")

### Step 1 — Load & verify

- Shape
- Columns
- Data types
- Null values
- Duplicate rows

In [ ]:
orders.shape

In [ ]:
orders.columns

In [ ]:
orders.dtypes

In [ ]:
orders.isnull().sum()

In [ ]:
orders.duplicated().sum()

### Step 2 — Nulls vs Order Status

In [ ]:
orders[orders["order_approved_at"].isnull()]["order_status"].value_counts()

In [ ]:
orders[orders["order_delivered_carrier_date"].isnull()]["order_status"].value_counts()

In [ ]:
orders[orders["order_delivered_customer_date"].isnull()]["order_status"].value_counts()

In [ ]:
orders[
    (orders["order_approved_at"].isnull()) &
    (orders["order_status"] == "delivered")
]

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])

In [ ]:
orders[
    (orders["order_approved_at"].isnull()) &
    (orders["order_status"] == "delivered")
][["order_purchase_timestamp", "order_delivered_carrier_date"]]


In [ ]:
purchase_carrier_days = orders["order_delivered_carrier_date"] - orders["order_purchase_timestamp"]


In [ ]:
purchase_carrier_days.max()

### Date Columns Datetime Conversion

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

In [ ]:
orders.dtypes

In [ ]:
invalid_delivery = orders[orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]]
len(invalid_delivery)

In [ ]:
invalid_carrier = orders[orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]]
len(invalid_carrier)

In [ ]:
orders["has_invalid_timeline"] = (
    (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]) |
    (orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"])
)

orders["has_invalid_timeline"].value_counts()

In [ ]:
orders["order_status"].value_counts()

In [ ]:
orders.columns

In [ ]:
import os

os.makedirs("../data/processed", exist_ok=True)
orders.to_csv("../data/processed/orders_cleaned.csv", index=False)

## Order_Items data set Cleaning

In [ ]:
order_items = pd.read_csv(r"C:\Users\MAINAK\Desktop\Data-Science\projects\MarketLens\data\raw\olist_order_items_dataset.csv")

#  shape and columns 

print("Shape : ",order_items.shape)
print("\n Columns : ",order_items.columns.tolist())


In [ ]:
#. Check missing Values 
print("\nCheck missing values : \n",order_items.isnull().sum())

# exact duplicates row Check 

print("\n Duplicates rows : \n ", order_items.duplicated().sum())

In [32]:
# Date _times change 
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

# Statistical summary of numerical columns
order_items[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [ ]:
print("Prices <= 0:", (order_items["price"] <= 0).sum())
print("Freight < 0:", (order_items["freight_value"] < 0).sum())
print("Freight == 0 (Free shipping):", (order_items["freight_value"] == 0).sum())

In [34]:
# 1. Composite key uniqueness check
composite_duplicates = order_items.duplicated(subset=["order_id", "order_item_id"]).sum()
print("Composite key duplicates (order_id + order_item_id):", composite_duplicates)

# 2. Referential integrity: Orphan items check against orders
orphan_items = (~order_items["order_id"].isin(orders["order_id"])).sum()
print("Orphan items count (orders me missing):", orphan_items)

Composite key duplicates (order_id + order_item_id): 0
Orphan items count (orders me missing): 0


In [35]:
# ensure processed directory exists
os.makedirs("../data/processed", exist_ok=True)

# save cleaned order_items
order_items.to_csv("../data/processed/order_items_cleaned.csv", index=False)